# IdiomBERT — Main-readiness evidence package

Goal: turn the **tokenizer-dependent flip** (QA↔BIO exact-match winner reverses by encoder) from a 2-encoder / 1-seed result into a Main-grade contribution.

## What this notebook produces
1. **Flip evidence** — 3 encoders × 2 systems × 3 seeds, full EN+ES+HI+TE combo. Breaks the n=2 confound (mBERT WordPiece, XLM-R SentencePiece, MuRIL WordPiece-Indic) and adds seed CIs.
2. **CI aggregation** — per-encoder QA vs BIO exact gap with across-seed std; confirms the flip is seed-stable.
3. **Mechanism analysis** — tokenizer → char-boundary reachability (no training); explains *why* QA's subtoken pointers beat BIO's word-snapped tagging under some tokenizers.
4. **C5 bonus** — Indonesian zero-shot falls out of every full-combo run (Indonesian in test_langs).

## Main-readiness checklist (the contribution clears Main when ALL true)
- [ ] Flip locked: bootstrap recompute == canonical `Full_evaluation.py` (run the lock cell).
- [ ] ≥3 encoders: QA-vs-BIO exact gap **sign differs by encoder**, seed-stable (across-seed std doesn't cross 0).
- [ ] Mechanism: tokenizer boundary-reachability rates explain the gap direction.
- [ ] Framing: claim is **encoder-dependent** (tokenizer = leading hypothesis), never "tokenizer proven cause" at n=3.

## Two-account split (you + friend, both free T4)
Both accounts run the SAME notebook; only change `SHARD` in the config cell.
- Owner: create Drive folder `IdiomatorRigor/`, share with friend (Editor). Friend: *Add shortcut to My Drive* (Colab can't write to "Shared with me").
- `SHARD='A'` (you) and `SHARD='B'` (friend) split the 18 jobs evenly (interleaved, so an account never gets all the slow ones).
- Both write to the SAME shared `IdiomatorRigor/flip/` — outputs never collide (per-job subdir). Whoever finishes last runs the aggregation + mechanism cells.

~18 runs × ~13–15 min each ≈ 4–5 h total ≈ 2–2.5 h per account.

## 0. GPU check (set Runtime → T4 GPU first)

In [ ]:
import os
assert os.system('nvidia-smi >/dev/null 2>&1') == 0, 'No GPU — Runtime → Change runtime type → T4 GPU'
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Clone / pull repo (repo root IS Research_And_Training — no subdir)

In [ ]:
from pathlib import Path
REPO = '/content/Idiomator_Research'
if Path(REPO).exists():
    !cd $REPO && git pull --ff-only
else:
    !git clone https://github.com/JustLetMeBeHello/Idiomator_Research.git $REPO
%cd $REPO
!pip install -q -r Requirements.txt

## 2. Config — mount Drive, set SHARD + FORCE

**Change `SHARD` only.** `'A'` on your account, `'B'` on your friend's. `FORCE=True` re-runs even if a job's `metrics.json` already exists (use after a code fix).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SHARD = 'A'          # <-- 'A' (you) or 'B' (friend)
FORCE = False        # <-- True to overwrite existing outputs (post code-fix rerun)

DRIVE_OUT = '/content/drive/MyDrive/IdiomatorRigor'
FLIP_DIR  = f'{DRIVE_OUT}/flip'
os.environ['DRIVE_OUT'] = DRIVE_OUT
Path(FLIP_DIR).mkdir(parents=True, exist_ok=True)
assert FLIP_DIR.startswith('/content/drive/'), 'persistence gate: output not on Drive'
print('SHARD =', SHARD, '| FORCE =', FORCE, '| FLIP_DIR =', FLIP_DIR)

## 3. Run the sharded flip matrix

3 encoders × {Joint(QA), BIO} × 3 seeds = 18 jobs. Each encoder uses its **published default LR** (mBERT/MuRIL 2e-5 Joint, XLM-R 1e-5 Joint; BIO 3.27e-5 all) — all other HP fixed, so only the encoder/tokenizer varies. Idempotent: skips a job whose `metrics.json` exists unless `FORCE`.

In [ ]:
ENCODERS = {
    'mbert': dict(model='bert-base-multilingual-cased', joint_lr='2e-5'),
    'xlmr':  dict(model='xlm-roberta-base',             joint_lr='1e-5'),
    'muril': dict(model='google/muril-base-cased',      joint_lr='2e-5'),
}
SEEDS = [42, 123, 7]
LANGS = 'English Spanish Hindi Telugu'
TEST_LANGS = 'English Spanish Hindi Telugu Indonesian'

# Build the full job list, then interleave-split by shard so neither account
# gets all the slow encoders.
JOBS = []
for enc, cfg in ENCODERS.items():
    for system in ('joint', 'bio'):
        for seed in SEEDS:
            JOBS.append((enc, system, seed, cfg))
mine = JOBS[0::2] if SHARD == 'A' else JOBS[1::2]
print(f'SHARD {SHARD}: {len(mine)} of {len(JOBS)} jobs')

for enc, system, seed, cfg in mine:
    outdir = f'{FLIP_DIR}/{enc}_{system}_s{seed}'
    Path(outdir).mkdir(parents=True, exist_ok=True)
    assert outdir.startswith('/content/drive/'), 'persistence gate'
    done = Path(f'{outdir}/metrics.json').exists()
    if done and not FORCE:
        print(f'✓ skip (exists): {enc}_{system}_s{seed}')
        continue
    print(f'\n===== {enc} {system} seed {seed} =====')
    if system == 'joint':
        cmd = (f"python Train_Join.py --model_name {cfg['model']} --output_dir {outdir} "
               f"--langs {LANGS} --test_langs {TEST_LANGS} "
               f"--epochs 7 --batch_size 32 --lr {cfg['joint_lr']} "
               f"--cls_loss_weight 0.3 --span_loss_weight 1.9 --seed {seed}")
    else:
        cmd = (f"python Ablations/BiO_Task_mBERT_train.py --model_name {cfg['model']} --output_dir {outdir} "
               f"--langs {LANGS} --test_langs {TEST_LANGS} "
               f"--epochs 6 --batch_size 32 --lr 3.27e-5 --o_weight 0.104 --seed {seed}")
    rc = os.system(f'{cmd} 2>&1 | tee -a {outdir}/console.log')
    print(f'  exit={rc>>8}')
print('\nSHARD', SHARD, 'done. Run aggregation once BOTH shards finished.')

## 4. Aggregate — the flip table with seed CIs

Run after BOTH accounts finish (all 18 dirs present in the shared Drive). Recomputes char-exact identically for QA and BIO (ignores the stored `span_exact_match`, which is token-level for Joint, char-level for BIO). Reports per-encoder QA−BIO gap as **mean ± across-seed std**; the flip is confirmed if the gap **sign differs by encoder** and the std doesn't cross 0.

In [ ]:
import json, statistics
from collections import defaultdict
load = lambda p: [json.loads(l) for l in open(p, encoding='utf-8')]
LANGS_ORDER = ['English', 'Spanish', 'Hindi', 'Telugu', 'Indonesian']

def exact_by_lang(path):
    agg = defaultdict(list)
    for r in load(path):
        gs, ge = r.get('span_start'), r.get('span_end')
        ps, pe = r.get('pred_span_start'), r.get('pred_span_end')
        agg[r['language']].append(int(ps is not None and gs is not None and ps == gs and pe == ge))
    return {l: sum(v)/len(v) for l, v in agg.items()}

# per (encoder, system, lang) -> list over seeds of exact mean
cell = defaultdict(lambda: defaultdict(list))
for enc in ENCODERS:
    for system in ('joint', 'bio'):
        for seed in SEEDS:
            p = f'{FLIP_DIR}/{enc}_{system}_s{seed}/test_predictions.jsonl'
            if not Path(p).exists():
                print('MISSING', p); continue
            for lang, ex in exact_by_lang(p).items():
                cell[(enc, system)][lang].append(ex)

print(f"\n{'encoder':7}{'lang':11}{'QA':>6}{'BIO':>7}{'gap':>8}{'±std':>7}  winner")
flip_summary = defaultdict(dict)
for enc in ENCODERS:
    for lang in LANGS_ORDER:
        qa = cell[(enc,'joint')].get(lang); bio = cell[(enc,'bio')].get(lang)
        if not qa or not bio: continue
        qm, bm = statistics.mean(qa), statistics.mean(bio)
        gaps = [a-b for a, b in zip(qa, bio)]
        g = statistics.mean(gaps)
        sd = statistics.pstdev(gaps) if len(gaps) > 1 else 0.0
        win = 'QA' if g > 0 else 'BIO'
        flip_summary[lang][enc] = (g, sd)
        print(f"{enc:7}{lang:11}{qm:>6.2f}{bm:>7.2f}{g:>+8.2f}{sd:>7.2f}  {win}")
    print()

print('FLIP CHECK (gap sign by encoder, per language):')
for lang in LANGS_ORDER:
    row = flip_summary.get(lang, {})
    if len(row) < 2: continue
    signs = {enc: ('QA' if g > 0 else 'BIO') for enc, (g, sd) in row.items()}
    flips = len(set(signs.values())) > 1
    detail = '  '.join(f"{e}:{signs[e]}({g:+.2f}±{sd:.2f})" for e, (g, sd) in row.items())
    print(f"  {lang:11} {'FLIPS' if flips else 'consistent':10} | {detail}")

## 5. Mechanism — tokenizer char-boundary reachability (no training)

Hypothesis for *why* the winner flips: BIO tags first-subtoken-of-word → its spans snap to **word** boundaries; QA predicts start/end at the **subtoken** level → it can hit boundaries inside a word that BIO cannot. Tokenizers differ in how often gold idiom boundaries land on word vs subtoken vs neither. This cell quantifies that per tokenizer — the mechanistic backing reviewers will want.

In [ ]:
from transformers import AutoTokenizer
GOLD = 'idioms_structured/Splits/test.jsonl'
gold = [r for r in load(GOLD) if r.get('span_start') is not None]

def reachability(tok_name):
    tk = AutoTokenizer.from_pretrained(tok_name)
    per = defaultdict(lambda: dict(word=0, subtok=0, none=0, n=0))
    for r in gold:
        s = r['sentence']; gs, ge = r['span_start'], r['span_end']; lang = r['language']
        enc = tk(s, return_offsets_mapping=True)
        offs, wids = enc['offset_mapping'], enc.word_ids()
        word_starts, word_ends, sub_starts, sub_ends = set(), set(), set(), set()
        prev = object()
        for i, (o, w) in enumerate(zip(offs, wids)):
            if w is None: continue
            sub_starts.add(o[0]); sub_ends.add(o[1])
            if w != prev: word_starts.add(o[0])
            # word end = end of the last subtoken whose next has a different/None wid
            nxt = wids[i+1] if i+1 < len(wids) else None
            if nxt != w: word_ends.add(o[1])
            prev = w
        c = per[lang]; c['n'] += 1
        if gs in word_starts and ge in word_ends:
            c['word'] += 1
        elif gs in sub_starts and ge in sub_ends:
            c['subtok'] += 1
        else:
            c['none'] += 1
    return per

TOKS = {'mbert': 'bert-base-multilingual-cased', 'xlmr': 'xlm-roberta-base', 'muril': 'google/muril-base-cased'}
print(f"{'tok':7}{'lang':11}{'word%(BIO-ok)':>14}{'subtok%(QA-only)':>17}{'none%':>8}")
for name, hf in TOKS.items():
    per = reachability(hf)
    for lang in LANGS_ORDER:
        c = per.get(lang)
        if not c or c['n'] == 0: continue
        n = c['n']
        print(f"{name:7}{lang:11}{100*c['word']/n:>13.1f}{100*c['subtok']/n:>17.1f}{100*c['none']/n:>8.1f}")
    print()
print('Read: higher subtok%% (QA-only-reachable) under a tokenizer predicts QA>BIO on exact for that tokenizer.')

## 6. (Optional) Lock vs canonical pipeline

Confirms the char-exact recompute matches `Full_evaluation.py`. Writes to a **separate** dir — never overwrites the canonical `results/pipeline_eval/pipeline_eval_results.json`.

In [ ]:
# Pick one encoder's seed-42 run to lock (e.g. xlmr).
ENC = 'xlmr'
QA  = f'{FLIP_DIR}/{ENC}_joint_s42/test_predictions.jsonl'
BIO = f'{FLIP_DIR}/{ENC}_bio_s42/test_predictions.jsonl'
A_qa, A_bio = exact_by_lang(QA), exact_by_lang(BIO)
os.system(f'python Evaluation/Full_evaluation.py --joint_preds {QA} --bio_preds {BIO} '
          f'--output_dir results/rigor_{ENC}_eval --n_bootstrap 10000')
d = json.load(open(f'results/rigor_{ENC}_eval/pipeline_eval_results.json'))
B_qa  = d['system_e_joint_end_to_end']['span_e2e']['exact']
B_bio = d['system_g_bio_tagger']['span_e2e']['exact']
ok = True
print(f"{'lang':11}{'QA(A)':>7}{'QA(B)':>7}   {'BIO(A)':>7}{'BIO(B)':>7}  verdict")
for l in LANGS_ORDER:
    qa_a, qa_b = A_qa.get(l,0), B_qa.get(l,0); bio_a, bio_b = A_bio.get(l,0), B_bio.get(l,0)
    p = abs(qa_a-qa_b) < 0.011 and abs(bio_a-bio_b) < 0.011; ok &= p
    print(f"{l:11}{qa_a:>7.2f}{qa_b:>7.2f}   {bio_a:>7.2f}{bio_b:>7.2f}  {'PASS' if p else 'FAIL'}")
print('\n' + ('✓ LOCKED' if ok else '✗ MISMATCH — investigate decode path'))